In [1]:
import torch
from torch import nn

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # Doesnt require transpose as its a single dimensions

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [6]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

In [7]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [8]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [9]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [16]:
import torch.nn as nn

class SimpleAttention(nn.Module):

    def __init__(self , d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
    
    def forward(self , x):
        query = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = query @ keys.T

        print("attn_scores:", attn_scores)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SimpleAttention(d_in, d_out)
print(sa_v1(inputs))

attn_scores: tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


# Detailed Explanation for the Self Attention Mechanism 

## Overview

This code implements a **basic self-attention mechanism** using PyTorch.
Self-attention allows each token in a sequence to **look at all other tokens and compute relationships** between them.

It is the **core building block of Transformer models** like GPT, BERT, and LLaMA.

The attention mechanism computes:

$$
Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:

* **Q** = Query matrix
* **K** = Key matrix
* **V** = Value matrix
* **d_k** = dimension of the key vector

---

# 1. Importing PyTorch Module

```python
import torch.nn as nn
```

This imports the **neural network module from PyTorch**, which provides:

* Layers
* Parameters
* Model structures

We inherit from `nn.Module` to build a custom layer.

---

# 2. Creating the Attention Class

```python
class SimpleAttention(nn.Module):
```

This defines a **custom neural network layer** implementing self-attention.

All transformer components like:

* Attention
* Feedforward networks
* Embeddings

are typically implemented as **`nn.Module` classes**.

---

# 3. Constructor (`__init__`)

```python
def __init__(self , d_in, d_out):
    super().__init__()
```

### Parameters

| Parameter | Meaning                    |
| --------- | -------------------------- |
| `d_in`    | input embedding dimension  |
| `d_out`   | output embedding dimension |

Example:

```
d_in = 512
d_out = 512
```

`super().__init__()` initializes the parent class (`nn.Module`).

---

# 4. Query, Key, Value Weight Matrices

```python
self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
self.W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
```

These matrices project the input embeddings into:

* **Query vectors**
* **Key vectors**
* **Value vectors**

### Shapes

| Matrix    | Shape           |
| --------- | --------------- |
| `W_query` | `(d_in, d_out)` |
| `W_key`   | `(d_in, d_out)` |
| `W_value` | `(d_in, d_out)` |

### Why we need them

The input embedding:

```
x → Query, Key, Value
```

Each projection learns **different semantic roles**.

```
Query → what am I looking for?
Key   → what do I contain?
Value → information to pass forward
```

---

### Why `requires_grad=False`?

Normally attention weights are **trainable**.

Here it is disabled just for **demonstration purposes**.

---

# 5. Forward Pass

```python
def forward(self , x):
```

The `forward()` function defines **how data flows through the layer**.

Input:

```
x = token embeddings
```

Example shape:

```
x.shape = (sequence_length , d_in)
```

Example:

```
6 tokens × 3 embedding size
```

---

# 6. Creating Queries

```python
query = x @ self.W_query
```

Matrix multiplication:

```
(sequence_len , d_in)
      ×
(d_in , d_out)
      ↓
(sequence_len , d_out)
```

Result:

```
Query matrix Q
```

Each token now has a **query vector**.

---

# 7. Creating Keys

```python
keys = x @ self.W_key
```

Same idea:

```
x → keys
```

Result:

```
Key matrix K
```

---

# 8. Creating Values

```python
values = x @ self.W_value
```

Result:

```
Value matrix V
```

---

# 9. Computing Attention Scores

```python
attn_scores = query @ keys.T
```

This computes **similarity between tokens**.

Shape calculation:

```
Q = (T , d)
K = (T , d)

Q @ K^T

(T , d) × (d , T) = (T , T)
```

Result:

```
attention score matrix
```

Example:

```
      token1 token2 token3 token4
token1   .     .      .      .
token2   .     .      .      .
token3   .     .      .      .
token4   .     .      .      .
```

Each value represents:

```
how much token i attends to token j
```

---

# 10. Printing Attention Scores

```python
print("attn_scores:", attn_scores)
```

This is for **debugging or visualization**.

---

# 11. Scaling the Scores

```python
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5, dim=-1
)
```

This implements the **scaled dot-product attention**.

### Why scaling?

Without scaling, large dot products cause:

```
softmax saturation
```

So we divide by:

[
\sqrt{d_k}
]

Where:

```
d_k = keys.shape[-1]
```

---

# 12. Softmax

Softmax converts scores into **probabilities**.

Example:

Before softmax

```
[2.1 , 0.4 , 1.2]
```

After softmax

```
[0.62 , 0.12 , 0.26]
```

Meaning:

```
token attends 62% to token1
```

---

# 13. Computing Context Vector

```python
context_vec = attn_weights @ values
```

Shape calculation:

```
(T , T) × (T , d)

↓

(T , d)
```

This produces a **weighted combination of value vectors**.

Each token now contains **information from all other tokens**.

---

# 14. Returning Output

```python
return context_vec
```

Output shape:

```
(sequence_length , d_out)
```

This is the **contextualized embedding**.

---

# 15. Setting Random Seed

```python
torch.manual_seed(123)
```

Ensures **reproducible results**.

---

# 16. Creating Attention Layer

```python
sa_v1 = SimpleAttention(d_in, d_out)
```

This initializes the attention module.

---

# 17. Running the Model

```python
print(sa_v1(inputs))
```

This:

1. Passes input tokens
2. Computes attention
3. Produces context vectors

---

# 18. Complete Attention Pipeline

```
Input Embeddings (x)
        │
        ▼
Linear projections
(Q , K , V)
        │
        ▼
Q × Kᵀ
        │
        ▼
Scale by √d
        │
        ▼
Softmax
        │
        ▼
Attention Weights
        │
        ▼
Weights × Values
        │
        ▼
Context Vectors
```

---

# 19. Limitations of This Implementation

This version is **simplified**.

Missing features used in real Transformers:

| Feature                  | Used in GPT |
| ------------------------ | ----------- |
| Multi-head attention     | ✔           |
| Causal masking           | ✔           |
| FlashAttention           | ✔           |
| KV cache                 | ✔           |
| RoPE positional encoding | ✔           |

---

# 20. Real Transformer Attention

Modern attention looks like:

```
Input
   │
Q,K,V projections
   │
Split into multiple heads
   │
Scaled Dot Product Attention
   │
Concatenate heads
   │
Linear projection
```

---

✅ **Key idea**

Self-attention allows each token to **learn relationships with all other tokens**, creating **context-aware embeddings**.

---
